# THERMOSCOPE · 01 Dataset exploration
Reads `outputs/dataset.csv` and `outputs/dataset_audit.json` (run `python -m thermoscope.run` first).

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('../outputs/dataset.csv'); audit = json.load(open('../outputs/dataset_audit.json'))
audit

In [ ]:
df.head(10)

In [ ]:
df.describe().T

## Data quality

In [ ]:
print('rows', len(df), '| missing', int(df.isna().sum().sum()), '| duplicates', int(df.duplicated(['fluid','Tr','Pr']).sum()))
pd.Series(audit['rejected'], name='rejected states')

## Phase distribution and samples per fluid

In [ ]:
names = {0: 'vapour', 1: 'liquid', 2: 'supercritical'}
df.phase.map(names).value_counts()

In [ ]:
df.assign(phase=df.phase.map(names)).pivot_table(index=['fluid','omega','split'], columns='phase', values='Z', aggfunc='count', fill_value=0)

## Distributions

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, c in zip(ax.ravel(), ['Tr', 'Pr', 'omega', 'Z', 'ar', 'hr', 'sr', 'gr']):
    a.hist(np.log10(df[c]) if c == 'Pr' else df[c], bins=40); a.set_title('log10 Pr' if c == 'Pr' else c)
plt.tight_layout()

## Correlation matrix

In [ ]:
cols = ['Tr', 'Pr', 'omega', 'Z', 'ar', 'hr', 'sr', 'gr', 'lnphi']
C = df[cols].assign(Pr=np.log(df.Pr)).corr()
plt.figure(figsize=(7, 6)); plt.imshow(C, cmap='coolwarm', vmin=-1, vmax=1); plt.xticks(range(len(cols)), cols); plt.yticks(range(len(cols)), cols); plt.colorbar(); C.round(2)

## State space coloured by phase

In [ ]:
plt.figure(figsize=(7, 5)); plt.scatter(df.Tr, df.Pr, c=df.phase, s=3, cmap='viridis'); plt.yscale('log'); plt.xlabel('Tr'); plt.ylabel('Pr')